# Crop Map: Normandy 23,04 × 23,04 km

Chín khối liền nhau, 10 band, tối đa 40 ngày thật/khối mùa 2018–2019.
Bật Internet; CPU đủ, không cần token hoặc nhãn. Dữ liệu ảnh khoảng 7,91 GiB
chưa gồm cache. Giữ phiên đủ dung lượng và lưu output trước khi kết thúc.
Notebook thu thập ảnh; chạy precompute_regions.py bằng checkpoint để tạo
classification cho web. Không biến nhãn hay ảnh nền bản đồ thành prediction.


In [ ]:
import importlib.util,subprocess,sys
needed={'numpy':'numpy>=1.26,<3','httpx':'httpx>=0.27,<1',
        'rasterio':'rasterio>=1.4,<2','pyproj':'pyproj>=3.6,<4'}
missing=[spec for name,spec in needed.items() if importlib.util.find_spec(name) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install',*missing])
from pathlib import Path
import json


## Code và cấu hình gốc

In [ ]:
Path('cog_data.py').write_text('"""Prepare a continuous, georeferenced Sentinel-2 cube before web inference.\n\nNo torch, web server, labels or credentials required. Completed scene files\nare atomic and reusable after interruption. Only completed cubes are published.\n"""\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom datetime import date\nimport hashlib\nimport fcntl\nimport json\nfrom pathlib import Path\nimport time\nimport tarfile\nfrom urllib.parse import urlsplit\n\nimport httpx\nimport numpy as np\nfrom pyproj import Transformer\nimport rasterio\nfrom rasterio.enums import Resampling\nfrom rasterio.features import geometry_mask\nfrom rasterio.transform import from_origin\nfrom rasterio.vrt import WarpedVRT\nfrom rasterio.warp import transform_geom\n\nAPI = \'https://earth-search.aws.element84.com/v1\'\nASSETS = [\'blue\',\'green\',\'red\',\'rededge1\',\'rededge2\',\'rededge3\',\'nir\',\'nir08\',\'swir16\',\'swir22\']\nBANDS = [\'B2\',\'B3\',\'B4\',\'B5\',\'B6\',\'B7\',\'B8\',\'B8A\',\'B11\',\'B12\']\nENV = dict(GDAL_DISABLE_READDIR_ON_OPEN=\'EMPTY_DIR\', CPL_VSIL_CURL_ALLOWED_EXTENSIONS=\'.tif\',\n           GDAL_HTTP_TIMEOUT=\'45\', GDAL_HTTP_CONNECTTIMEOUT=\'15\',\n           GDAL_HTTP_MAX_RETRY=\'2\', GDAL_HTTP_RETRY_DELAY=\'2\')\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix+\'.tmp\')\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False))\n    temporary.replace(path)\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda:stream.read(8*1024**2), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef fingerprint(value):\n    return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()\n\n\ndef grid(config):\n    return from_origin(*config[\'origin\'],10,10)\n\n\ndef rectangle(config, row=0, col=0, height=None, width=None):\n    height = config[\'shape\'][0] if height is None else height\n    width = config[\'shape\'][1] if width is None else width\n    west,north = config[\'origin\'][0]+col*10, config[\'origin\'][1]-row*10\n    convert = Transformer.from_crs(config[\'crs\'],4326,always_xy=True)\n    ring = [(west,north),(west+width*10,north),(west+width*10,north-height*10),\n            (west,north-height*10),(west,north)]\n    return dict(type=\'Polygon\',coordinates=[[list(convert.transform(*point)) for point in ring]])\n\n\ndef validate_config(c):\n    if not isinstance(c[\'id\'],int) or c[\'id\'] < 900000:\n        raise ValueError(\'Use a distinct COG region ID >= 900000.\')\n    if not c[\'slug\'] or any(x not in \'abcdefghijklmnopqrstuvwxyz0123456789-\' for x in c[\'slug\']):\n        raise ValueError(\'Unsafe region slug\')\n    if any(not isinstance(n,int) or n<24 or n>1536 or n%24 for n in c[\'shape\']):\n        raise ValueError(\'Shape must be 24-pixel multiples, at most 1536 per side.\')\n    if not 1 <= c[\'minimum_observations\'] <= c[\'observations\'] <= 60:\n        raise ValueError(\'Require 1 <= minimum_observations <= observations <= 60.\')\n    if not 1<=c[\'workers\']<=8:\n        raise ValueError(\'Use 1–8 workers.\')\n    if any(not 0<=c[key]<=1 for key in [\'minimum_clear_fraction\',\'minimum_native_fraction\']):\n        raise ValueError(\'Quality fractions must be between 0 and 1.\')\n    if not 1<=c[\'minimum_clear_observations\']<=c[\'minimum_observations\']:\n        raise ValueError(\'Invalid per-pixel clear observation requirement.\')\n    if not 24<=c[\'default_selection_pixels\']<=min(c[\'shape\']) or c[\'default_selection_pixels\']%24:\n        raise ValueError(\'Invalid default selection size.\')\n    if date.fromisoformat(c[\'start_date\'])>=date.fromisoformat(c[\'end_date\']):\n        raise ValueError(\'Invalid date interval\')\n\n\ndef search(config):\n    ring = np.array(rectangle(config)[\'coordinates\'][0])\n    query = dict(collections=[\'sentinel-2-c1-l2a\',\'sentinel-2-l2a\'],limit=100,\n                 bbox=[float(ring[:,0].min()),float(ring[:,1].min()),\n                       float(ring[:,0].max()),float(ring[:,1].max())],\n                 datetime=config[\'start_date\']+\'T00:00:00Z/\'+config[\'end_date\']+\'T23:59:59Z\')\n    found = {}\n    with httpx.Client(timeout=60) as client:\n        for _ in range(30):\n            for attempt in range(3):\n                response=client.post(API+\'/search\',json=query)\n                if response.status_code not in (429,500,502,503,504) or attempt==2:break\n                time.sleep(2)\n            response.raise_for_status()\n            body=response.json()\n            for item in body[\'features\']:\n                if all(a in item.get(\'assets\',{}) for a in ASSETS+[\'scl\']):\n                    found[(item[\'collection\'],item[\'id\'])]=item\n            following=next((x for x in body.get(\'links\',[]) if x.get(\'rel\')==\'next\'),None)\n            if not following:break\n            query=following[\'body\']\n        else:raise RuntimeError(\'Too many catalog pages\')\n    # Full-region footprint coverage, and one real date per observation.\n    tiles={}\n    for item in found.values():\n        footprint=transform_geom(\'EPSG:4326\',config[\'crs\'],item[\'geometry\'])\n        inside=geometry_mask([footprint],out_shape=tuple(config[\'shape\']),transform=grid(config),invert=True)\n        if inside.all():tiles.setdefault(item[\'properties\'][\'grid:code\'],[]).append(item)\n    if not tiles:raise RuntimeError(\'Choose a region covered by one Sentinel-2 tile.\')\n    tile,entries=max(tiles.items(),key=lambda pair:len({x[\'properties\'][\'datetime\'][:10] for x in pair[1]}))\n    chosen={}\n    def rank(item):\n        p=item[\'properties\']\n        return (item[\'collection\']==\'sentinel-2-c1-l2a\',float(p.get(\'s2:processing_baseline\',0)),p.get(\'updated\',\'\'))\n    for item in entries:\n        day=item[\'properties\'][\'datetime\'][:10]\n        if day not in chosen or rank(item)>rank(chosen[day]):chosen[day]=item\n    return dict(tile=tile,features=sorted(chosen.values(),key=lambda x:x[\'properties\'][\'datetime\']))\n\n\ndef read_asset(item,key,config):\n    # Successful HTTP status can still carry a truncated TIFF byte range.\n    # Reopening with a cleared per-URL cache retries the actual band read.\n    for attempt in range(3):\n        try:\n            with rasterio.Env(CPL_VSIL_CURL_NON_CACHED=\'/vsicurl/\'+item[\'assets\'][key][\'href\']):\n                return _read_asset_once(item,key,config)\n        except rasterio.errors.RasterioIOError:\n            if attempt==2:raise\n            time.sleep(1+attempt)\n\n\ndef _read_asset_once(item,key,config):\n    asset=item[\'assets\'][key]\n    parsed=urlsplit(asset[\'href\'])\n    if (parsed.scheme!=\'https\' or not parsed.hostname or not parsed.hostname.endswith(\'.amazonaws.com\')\n            or not parsed.path.endswith(\'.tif\') or asset.get(\'storage:requester_pays\')):\n        raise ValueError(\'Expected a public Sentinel-2 HTTPS COG asset.\')\n    info=asset.get(\'raster:bands\',[{}])[0]\n    with rasterio.Env(**ENV),rasterio.open(asset[\'href\']) as source:\n        nodata=info.get(\'nodata\',source.nodata)\n        if nodata is None:raise ValueError(\'Missing nodata metadata\')\n        with WarpedVRT(source,crs=config[\'crs\'],transform=grid(config),\n                       height=config[\'shape\'][0],width=config[\'shape\'][1],\n                       src_nodata=nodata,nodata=nodata,dtype=\'float32\',\n                       resampling=Resampling.nearest if key==\'scl\' else Resampling.bilinear) as vrt:\n            raw=vrt.read(1)\n    native=np.isfinite(raw)&(raw!=nodata)\n    if key==\'scl\':return raw.astype(np.uint8),native\n    if \'scale\' not in info:raise ValueError(\'Missing reflectance scale metadata\')\n    image=(raw*np.float32(info[\'scale\'])+np.float32(info.get(\'offset\',0)))*np.float32(10000)\n    image[~native]=0\n    return image,native\n\n\ndef save_scene(path,**values):\n    temporary=path.with_suffix(\'.tmp\')\n    with temporary.open(\'wb\') as stream:np.savez_compressed(stream,**values)\n    temporary.replace(path)\n\n\ndef select_dates(items,assessments,config):\n    eligible=[x for x in items if assessments[x[\'id\']][\'clear_fraction\']>=config[\'minimum_clear_fraction\']\n              and assessments[x[\'id\']][\'native_fraction\']>=config[\'minimum_native_fraction\']]\n    if len(eligible)<config[\'minimum_observations\']:\n        raise RuntimeError(f\'Only {len(eligible)} suitable dates; minimum is {config["minimum_observations"]}. \'\n                           \'Keep the quality thresholds explicit; do not invent observations.\')\n    count=min(config[\'observations\'],len(eligible))\n    start=date.fromisoformat(config[\'start_date\']).toordinal()\n    end=date.fromisoformat(config[\'end_date\']).toordinal()\n    remaining=list(eligible);chosen=[]\n    # Allocate across the entire actual season; quality is the tie-breaker.\n    for target in np.linspace(start,end,count):\n        best=min(remaining,key=lambda x:(abs(date.fromisoformat(x[\'properties\'][\'datetime\'][:10]).toordinal()-target),\n                    -assessments[x[\'id\']][\'clear_fraction\'],x[\'id\']))\n        remaining.remove(best);chosen.append(best)\n    return sorted(chosen,key=lambda x:x[\'properties\'][\'datetime\'])\n\n\ndef prepare(config,data_dir):\n    validate_config(config)\n    data_dir=Path(data_dir)\n    output=data_dir/\'cog\'/config[\'slug\']\n    output.mkdir(parents=True,exist_ok=True)\n    effective=output/\'config.json\'\n    semantic={k:v for k,v in config.items() if k!=\'workers\'}\n    if effective.exists() and {k:v for k,v in json.loads(effective.read_text()).items() if k!=\'workers\'}!=semantic:\n        raise ValueError(\'Configuration changed. Use a new slug to keep datasets separate.\')\n    write_json(effective,config)\n    def emit(stage,**values):\n        row=dict(stage=stage,**values)\n        write_json(output/\'progress.json\',row)\n        print(json.dumps(row),flush=True)\n    report_path=output/\'report.json\'\n    published=data_dir/\'cog/catalog.json\'\n    if report_path.exists() and published.exists():\n        report=json.loads(report_path.read_text())\n        entries=json.loads(published.read_text())[\'patches\']\n        if any(p[\'id\']==config[\'id\'] and p.get(\'dataset_version\')==report[\'dataset_version\'] for p in entries):\n            if all((output/name).is_file() and (output/name).stat().st_size==s[\'bytes\']\n                   and digest(output/name)==s[\'sha256\'] for name,s in report[\'files\'].items()):\n                emit(\'ready\',region_id=config[\'id\'],area_ha=report[\'area_ha\'],observations=report[\'observations\'],cached=True)\n                return report\n    started=time.monotonic()\n    catalog_file=output/\'catalog-source.json\'\n    if not catalog_file.exists():write_json(catalog_file,search(config))\n    catalog=json.loads(catalog_file.read_text());items=catalog[\'features\']\n    emit(\'catalog\',dates=len(items),tile=catalog[\'tile\'],shape=config[\'shape\'])\n    quality=output/\'cache-quality\';quality.mkdir(exist_ok=True)\n    scenes=output/\'cache-scenes\';scenes.mkdir(exist_ok=True)\n    assessments={}\n    def quality_date(item):\n        path=quality/(item[\'id\']+\'.npz\')\n        if not path.exists():\n            scl,native=read_asset(item,\'scl\',config)\n            save_scene(path,scl=scl,native=native)\n        with np.load(path,allow_pickle=False) as d:\n            native=d[\'native\']&(d[\'scl\']!=1)\n            clear=np.isin(d[\'scl\'],[4,5,6])&native\n            return dict(id=item[\'id\'],date=item[\'properties\'][\'datetime\'][:10],\n                        clear_fraction=float(clear.mean()),native_fraction=float(native.mean()))\n    errors=[]\n    with ThreadPoolExecutor(max_workers=config[\'workers\']) as pool:\n        futures={pool.submit(quality_date,item):item for item in items}\n        for future in as_completed(futures):\n            item=futures[future]\n            try:assessments[item[\'id\']]=future.result()\n            except Exception as error:\n                errors.append(item[\'id\']);emit(\'quality_error\',id=item[\'id\'],error=str(error))\n            if (len(assessments)+len(errors))%8==0:\n                emit(\'quality\',completed=len(assessments),total=len(items),errors=len(errors))\n    if errors:raise RuntimeError(f\'{len(errors)} quality reads failed. Run the same command to resume.\')\n    selected=select_dates(items,assessments,config)\n    selection=dict(items=[x[\'id\'] for x in selected],assessments=assessments)\n    write_json(output/\'selection.json\',selection)\n    emit(\'selected\',observations=len(selected),dates=[x[\'properties\'][\'datetime\'][:10] for x in selected])\n    for i,item in enumerate(selected):\n        path=scenes/(item[\'id\']+\'.npz\')\n        if not path.exists():\n            bands=[None]*10\n            with np.load(quality/(item[\'id\']+\'.npz\'),allow_pickle=False) as q:\n                native=q[\'native\']&(q[\'scl\']!=1);scl=q[\'scl\'].copy()\n            with ThreadPoolExecutor(max_workers=config[\'workers\']) as pool:\n                pending={pool.submit(read_asset,item,key,config):b for b,key in enumerate(ASSETS)}\n                try:\n                    for future in as_completed(pending):\n                        b=pending[future];bands[b],valid=future.result();native&=valid\n                except Exception:\n                    for f in pending:f.cancel()\n                    raise\n            save_scene(path,image=np.stack(bands),native=native,scl=scl)\n        emit(\'imagery\',completed=i+1,total=len(selected),date=item[\'properties\'][\'datetime\'][:10])\n    dates=[x[\'properties\'][\'datetime\'][:10] for x in selected]\n    doy=[date.fromisoformat(d).timetuple().tm_yday for d in dates]\n    order=np.argsort(doy,kind=\'stable\').tolist()\n    cube_path=output/\'observations.npy\'\n    temporary=output/\'observations.pending.npy\'\n    cube=np.lib.format.open_memmap(temporary,mode=\'w+\',dtype=\'float32\',shape=(len(selected),10,*config[\'shape\']))\n    valid=np.ones(config[\'shape\'],dtype=bool)\n    clear_count=np.zeros(config[\'shape\'],dtype=np.uint16)\n    for i,item in enumerate(selected):\n        with np.load(scenes/(item[\'id\']+\'.npz\'),allow_pickle=False) as d:\n            if d[\'image\'].shape!=(10,*config[\'shape\']) or not np.isfinite(d[\'image\']).all():\n                raise ValueError(\'Corrupt scene cache\')\n            cube[i]=d[\'image\'];valid&=d[\'native\']\n            clear_count+=np.isin(d[\'scl\'],[4,5,6])&d[\'native\']\n    cube.flush();del cube\n    temporary.replace(cube_path)\n    valid&=clear_count>=config[\'minimum_clear_observations\']\n    if not valid.any():raise RuntimeError(\'No pixels meet the native/clear observation policy\')\n    np.save(output/\'valid-mask.npy\',valid,allow_pickle=False)\n    np.save(output/\'clear-count.npy\',clear_count,allow_pickle=False)\n    # One real low-cloud acquisition for the full-region preview.\n    best=max(range(len(selected)),key=lambda i:assessments[selected[i][\'id\']][\'clear_fraction\'])\n    h,w=config[\'shape\'];side=config[\'default_selection_pixels\']\n    image_hash=digest(cube_path);mask_hash=digest(output/\'valid-mask.npy\')\n    version=fingerprint(dict(image=image_hash,valid=mask_hash,config=semantic))\n    patch=dict(id=config[\'id\'],name=config[\'name\'],region=\'Normandy\',source=\'Sentinel-2 COG\',\n               split=\'external\',season=config[\'season\'],crs=config[\'crs\'],origin=config[\'origin\'],\n               shape=config[\'shape\'],geometry=rectangle(config),\n               default_geometry=rectangle(config,(h-side)//2,(w-side)//2,side,side),\n               image_path=f\'cog/{config["slug"]}/observations.npy\',\n               valid_mask_path=f\'cog/{config["slug"]}/valid-mask.npy\',\n               frame_order=order,doy=[doy[i] for i in order],dates=dates,\n               acquisition_count=len(selected),model_observations=len(selected),\n               preview_index=best,preview_date=dates[best],first_date=min(dates),last_date=max(dates),\n               max_analysis_windows=config[\'max_analysis_windows\'],sha256=image_hash,dataset_version=version)\n    gaps=np.diff([date.fromisoformat(d).toordinal() for d in dates])\n    report=dict(source=API,accuracy_validated=False,region_id=config[\'id\'],area_ha=h*w*.01,\n                observations=len(selected),dates=dates,band_order=BANDS,frame_order=order,\n                max_gap_days=int(max(gaps,default=0)),valid_fraction=float(valid.mean()),\n                minimum_clear_observations=config[\'minimum_clear_observations\'],\n                cloud_policy=\'Whole-date clear fraction threshold; remaining cloudy reflectances retained. No imputation.\',\n                reflectance=\'asset scale/offset then x10000; bilinear bands, nearest SCL\',\n                dataset_version=version,elapsed_seconds=round(time.monotonic()-started,2),\n                files={name:dict(bytes=(output/name).stat().st_size,sha256=digest(output/name))\n                       for name in [\'observations.npy\',\'valid-mask.npy\',\'clear-count.npy\']},\n                provenance=selected)\n    write_json(output/\'report.json\',report)\n    # Atomic publication: the app never advertises an unfinished cube.\n    published=data_dir/\'cog/catalog.json\'\n    with (published.parent/\'catalog.lock\').open(\'a\') as lock:\n        fcntl.flock(lock,fcntl.LOCK_EX)\n        previous=json.loads(published.read_text()) if published.exists() else dict(patches=[])\n        previous[\'patches\']=[p for p in previous[\'patches\'] if p[\'id\']!=patch[\'id\']]+[patch]\n        previous[\'season\']=config[\'season\']\n        write_json(published,previous)\n    emit(\'ready\',region_id=config[\'id\'],area_ha=h*w*.01,observations=len(selected),valid_fraction=report[\'valid_fraction\'])\n    return report\n\n\ndef export_dataset(data_dir,destination,include_cache=False):\n    """Completed deploy data, or a resumable preparation checkpoint, as tar.gz."""\n    data_dir=Path(data_dir);destination=Path(destination)\n    destination.parent.mkdir(parents=True,exist_ok=True)\n    temporary=destination.with_suffix(destination.suffix+\'.part\')\n    files=[]\n    for path in (data_dir/\'cog\').rglob(\'*\'):\n        if not path.is_file() or path.is_symlink() or path.name.endswith((\'.tmp\',\'.part\')) or \'.pending.\' in path.name:\n            continue\n        if not include_cache and any(p.startswith(\'cache-\') for p in path.parts):continue\n        files.append(path)\n    if not files:raise ValueError(\'No COG data to export\')\n    with tarfile.open(temporary,\'w:gz\') as archive:\n        for path in sorted(files):archive.add(path,arcname=path.relative_to(data_dir).as_posix(),recursive=False)\n    temporary.replace(destination)\n    destination.with_suffix(destination.suffix+\'.sha256\').write_text(f\'{digest(destination)}  {destination.name}\\n\')\n    return destination\n\n\ndef restore_dataset(source,data_dir):\n    """Support both a Kaggle auto-extracted directory and an archive."""\n    import shutil\n    source=Path(source);data_dir=Path(data_dir)\n    if (data_dir/\'cog\').exists():raise ValueError(\'Restore only into a fresh data directory\')\n    data_dir.mkdir(parents=True,exist_ok=True)\n    if source.is_dir():\n        if not (source/\'cog\').is_dir():raise ValueError(\'Choose the directory containing cog/\')\n        shutil.copytree(source/\'cog\',data_dir/\'cog\')\n    else:\n        with tarfile.open(source) as archive:\n            for member in archive.getmembers():\n                path=(data_dir/member.name).resolve()\n                if not path.is_relative_to(data_dir.resolve()) or not (member.isfile() or member.isdir()):\n                    raise ValueError(\'Unsafe archive member\')\n            archive.extractall(data_dir,filter=\'data\')\n')
Path('prepare_regions.py').write_text('"""Prepare nine adjacent cubes, then publish one continuous virtual region.\n\nEach acquisition block remains independently resumable. No duplicate 8 GiB\nimage mosaic is created: inference routes native 24-pixel windows to a cube.\n"""\nimport argparse\nfrom copy import deepcopy\nfrom concurrent.futures import ProcessPoolExecutor,as_completed\nimport json\nfrom pathlib import Path\nimport time\nfrom multiprocessing import get_context\n\nimport httpx\nimport rasterio\n\nfrom cog_data import fingerprint, prepare, rectangle, write_json\n\nROOT = Path(__file__).resolve().parent\n\n\ndef block_configs(config):\n    base = config[\'block\']\n    size = base[\'shape\'][0]\n    for row in range(config[\'rows\']):\n        for col in range(config[\'cols\']):\n            block = deepcopy(base)\n            block.update(id=config[\'first_block_id\'] + row*config[\'cols\'] + col,\n                         slug=f"{config[\'slug\']}-r{row}-c{col}",\n                         name=f"Normandy block {row+1}/{col+1}",\n                         origin=[config[\'origin\'][0]+col*size*10,\n                                 config[\'origin\'][1]-row*size*10])\n            reuse = config.get(\'reuse\', {}).get(f\'{row},{col}\')\n            if reuse:\n                block = json.loads((ROOT/reuse).read_text())\n                expected=[config[\'origin\'][0]+col*size*10,config[\'origin\'][1]-row*size*10]\n                if block[\'shape\']!=base[\'shape\'] or block[\'origin\']!=expected or block[\'crs\']!=base[\'crs\']:\n                    raise ValueError(\'Reused cube does not align with the mosaic grid\')\n            yield row, col, block\n\n\ndef publish(config, data):\n    path=Path(data)/\'cog/catalog.json\'\n    catalog=json.loads(path.read_text())\n    entries={p[\'id\']:p for p in catalog[\'patches\']}\n    parts=[]\n    size=config[\'block\'][\'shape\'][0]\n    for row,col,c in block_configs(config):\n        if c[\'id\'] not in entries:raise ValueError(f"Block {c[\'id\']} is not ready")\n        p=entries[c[\'id\']]\n        if p[\'origin\']!=c[\'origin\'] or p[\'shape\']!=c[\'shape\'] or p[\'crs\']!=c[\'crs\']:\n            raise ValueError(\'Published block is not aligned\')\n        parts.append(dict(id=p[\'id\'],row=row*size,col=col*size))\n    shape=[config[\'rows\']*size,config[\'cols\']*size]\n    dates=sorted({d for part in parts for d in entries[part[\'id\']][\'dates\']})\n    counts=[entries[p[\'id\']][\'acquisition_count\'] for p in parts]\n    patch=dict(id=config[\'id\'],name=config[\'name\'],region=\'Normandy\',source=\'Sentinel-2 COG\',\n               split=\'external\',season=config[\'block\'][\'season\'],crs=config[\'block\'][\'crs\'],\n               origin=config[\'origin\'],shape=shape,components=parts,\n               acquisition_count=min(counts),observation_range=[min(counts),max(counts)],\n               model_observations=min(counts),dates=dates,first_date=min(dates),last_date=max(dates),\n               preview_date=\'Per block\',max_analysis_windows=256,\n               dataset_version=fingerprint([entries[p[\'id\']][\'dataset_version\'] for p in parts]))\n    patch[\'geometry\']=rectangle(patch)\n    patch[\'default_geometry\']=rectangle(patch,shape[0]//2-96,shape[1]//2-96,192,192)\n    # Hide internal blocks from the dropdown while keeping old explicit IDs usable.\n    for part in parts:entries[part[\'id\']][\'mosaic_parent\']=config[\'id\']\n    catalog[\'patches\']=[patch]+[p for p in entries.values() if p[\'id\']!=patch[\'id\']]\n    write_json(path,catalog)\n    write_json(Path(data)/\'cog\'/config[\'slug\']/\'report.json\',\n               dict(region_id=patch[\'id\'],area_km2=shape[0]*shape[1]/10000,\n                    blocks=len(parts),observation_range=patch[\'observation_range\'],\n                    dataset_version=patch[\'dataset_version\'],components=parts))\n    return patch\n\n\ndef acquire(entry,data):\n    row,col,block=entry\n    print(json.dumps(dict(stage=\'block\',row=row,col=col,id=block[\'id\'])),flush=True)\n    for attempt in range(3):\n        try:\n            return prepare(block,data)\n        except (rasterio.errors.RasterioIOError,httpx.HTTPError) as error:\n            if attempt==2:raise\n            print(json.dumps(dict(stage=\'block_retry\',id=block[\'id\'],error=str(error))),flush=True)\n            time.sleep(2)\n\n\ndef run(config,data,parallel_blocks=2):\n    # Bound both parallel regions and the four HTTP readers inside each region.\n    # Separate processes also isolate GDAL\'s global HTTP cache between blocks.\n    errors=[]\n    if parallel_blocks not in (1,2,3):raise ValueError(\'Use 1–3 parallel acquisition blocks\')\n    with ProcessPoolExecutor(max_workers=parallel_blocks,mp_context=get_context(\'spawn\')) as pool:\n        futures={pool.submit(acquire,entry,data):entry[2][\'id\'] for entry in block_configs(config)}\n        for future in as_completed(futures):\n            try:future.result()\n            except Exception as error:\n                errors.append(futures[future])\n                print(json.dumps(dict(stage=\'block_error\',id=futures[future],error=str(error))),flush=True)\n    if errors:raise RuntimeError(f\'Blocks {errors} failed; rerun the same command to resume.\')\n    patch=publish(config,data)\n    print(json.dumps(dict(stage=\'mosaic_ready\',id=patch[\'id\'],shape=patch[\'shape\'])),flush=True)\n    return patch\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--config\',type=Path,default=ROOT/\'preparation/regions_config.json\')\n    parser.add_argument(\'--data\',type=Path,default=ROOT/\'data\')\n    parser.add_argument(\'--plan\',action=\'store_true\')\n    parser.add_argument(\'--parallel-blocks\',type=int,default=2,choices=(1,2,3))\n    args=parser.parse_args();config=json.loads(args.config.read_text())\n    if args.plan:\n        print(json.dumps([dict(row=r,col=c,config=b) for r,c,b in block_configs(config)],indent=2))\n    else:run(config,args.data,args.parallel_blocks)\n')
Path("preparation").mkdir(exist_ok=True)
Path('preparation/cog_config.json').write_text('{\n  "id": 900001,\n  "slug": "normandy-2018-2019",\n  "name": "Normandy · 7.68 × 7.68 km",\n  "season": "2018–2019",\n  "start_date": "2018-10-01",\n  "end_date": "2019-09-30",\n  "crs": "EPSG:32630",\n  "origin": [688212.3644586978, 5468200.621113034],\n  "shape": [768, 768],\n  "observations": 40,\n  "minimum_observations": 30,\n  "minimum_clear_fraction": 0.5,\n  "minimum_native_fraction": 0.99,\n  "minimum_clear_observations": 3,\n  "default_selection_pixels": 192,\n  "max_analysis_windows": 256,\n  "workers": 4\n}\n')
Path('preparation/regions_config.json').write_text('{\n  "id": 900010,\n  "slug": "normandy-expanded-2018-2019",\n  "name": "Normandy \\u00b7 23.04 \\u00d7 23.04 km",\n  "rows": 3,\n  "cols": 3,\n  "origin": [\n    680532.3644586978,\n    5468200.621113034\n  ],\n  "first_block_id": 900020,\n  "reuse": {\n    "0,1": "preparation/cog_config.json"\n  },\n  "block": {\n    "id": 900001,\n    "slug": "normandy-2018-2019",\n    "name": "Normandy \\u00b7 7.68 \\u00d7 7.68 km",\n    "season": "2018\\u20132019",\n    "start_date": "2018-10-01",\n    "end_date": "2019-09-30",\n    "crs": "EPSG:32630",\n    "origin": [\n      688212.3644586978,\n      5468200.621113034\n    ],\n    "shape": [\n      768,\n      768\n    ],\n    "observations": 40,\n    "minimum_observations": 30,\n    "minimum_clear_fraction": 0.5,\n    "minimum_native_fraction": 0.99,\n    "minimum_clear_observations": 3,\n    "default_selection_pixels": 192,\n    "max_analysis_windows": 256,\n    "workers": 4\n  }\n}')
from prepare_regions import run
from cog_data import export_dataset,restore_dataset
CONFIG=json.loads(Path('preparation/regions_config.json').read_text())
IS_KAGGLE=Path('/kaggle/working').is_dir()
DATA=Path('/kaggle/temp/crop-map-regions/data') if IS_KAGGLE else Path('/content/crop-map-regions/data')
EXPORT=Path('/kaggle/working/submission') if IS_KAGGLE else Path('/content/submission')
DATA.mkdir(parents=True,exist_ok=True);EXPORT.mkdir(parents=True,exist_ok=True)
print('Scratch:',DATA,'Output:',EXPORT)


## Phục hồi tùy chọn

RESTORE_PATH là archive của lần trước hoặc thư mục đã giải nén chứa cog/.
Chỉ phục hồi vào scratch mới. Giữ đúng cấu hình để tái sử dụng ngày hoàn tất.


In [ ]:
RESTORE_PATH=None
if RESTORE_PATH:restore_dataset(RESTORE_PATH,DATA)


## Thu thập hai khối song song; lỗi mạng thì chạy lại cell

In [ ]:
region=run(CONFIG,DATA)
print('Ready:',region['name'],region['shape'])


## Lưu dữ liệu

Sau hoàn tất để INCLUDE_CACHE=False. Nếu phải dừng sớm, chọn True để mang
theo cache và chạy tiếp phiên sau. Gói lớn; kiểm tra dung lượng output của
phiên đang dùng. Giữ archive trước khi tắt phiên. Có thể đặt EXPORT lên
Google Drive đã mount trên Colab. Giải nén vào crop-map/data trên máy chuẩn
bị, rồi chạy precompute_regions.py và package_deploy.py --predictions-only.


In [ ]:
INCLUDE_CACHE=False
archive=export_dataset(DATA,EXPORT/'normandy-regions.tar.gz',include_cache=INCLUDE_CACHE)
print(archive,round(archive.stat().st_size/1024**3,2),'GiB')
